# Laboratorio 7 - Spark MLlib · Persona C
### Estadística descriptiva y perfiles de clusters
CC3066 Data Science, UVG, Semestre II 2026.

Este borrador cubre:

- **Ejercicio 2 completo:** estadísticas de salario, edad, antigüedad y horas para la población analítica de 2025, y las cinco preguntas de exploración con evidencia gráfica.
- **Ejercicio 4 (interpretación):** tamaño, perfil numérico (en escala original) y composición de cada cluster elegido por Persona B, gráficas de perfiles y nombre/descripción de cada cluster.

**Requisitos previos:** `parquet/prep_2025/` y `parquet/perfiles_2025/` (notebook de B) y los diccionarios en `working_dir/raw/diccionarios/`.

**Reglas que se respetan:** todas las estadísticas se calculan en Spark sobre **todos** los registros de 2025; a pandas solo pasan tablas agregadas o una muestra de hasta 5,000 filas para la dispersión; análisis no ponderado; los salarios extremos se conservan.

## 0. Configuración

Igual a la de A y B, solo con lo que usa esta parte. En el notebook final se fusiona con la sección 0.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from IPython.display import display, Markdown

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

SEED = 42
MAX_FILAS_GRAFICO = 5_000   # tope de filas que se transfieren a pandas para dibujar

BASE_DIR          = Path(os.environ.get("LAB7_BASE", "/opt/app/working_dir"))
DICC_DIR          = BASE_DIR / "raw" / "diccionarios"
PARQUET_DIR       = BASE_DIR / "parquet"
PREP_2025_DIR     = PARQUET_DIR / "prep_2025"
PERFILES_2025_DIR = PARQUET_DIR / "perfiles_2025"
FIG_DIR           = BASE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

VARS_NUM = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]
VARS_CAT = ["categoria_ocupacional", "nivel_educativo", "dominio"]
VARIABLES_CAT = {"P03A03A": "nivel_educativo", "P05C16": "categoria_ocupacional", "DOMINIO": "dominio"}
PERIODOS_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]

ETIQUETAS_NUM = {"salario_mensual": "Salario mensual (Q)", "edad": "Edad (años)",
                 "antiguedad": "Antigüedad (años)", "horas_semanales": "Horas semanales"}
ETIQUETAS_CAT = {"categoria_ocupacional": "Categoría ocupacional", "nivel_educativo": "Nivel educativo",
                 "dominio": "Dominio"}

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

In [ ]:
spark = (
    SparkSession.builder
    .appName("lab7-spark-mllib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    # arrow apagado igual que en A y B
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
assert spark.version.startswith("3.5"), "El laboratorio requiere Spark 3.5.x"

### 0.1 Funciones de apoyo

Funciones pequeñas y sin lectura de archivos, para reutilizarlas en todo el notebook y probarlas con datos sintéticos en el anexo.

- `resumen_numerico`: n, media, mediana, desviación estándar, mínimo, máximo, P25, P75 y P95 en **una sola pasada** por Spark, global o por grupo. Los percentiles son **exactos** (`percentile` de Spark SQL, con interpolación lineal), no aproximados.
- `composicion`: conteo y porcentaje por categoría, global o dentro de cada grupo.
- `histograma`: conteos por intervalo calculados en Spark con todos los registros (escala lineal o log10); a pandas solo pasa la tabla de intervalos.
- `etiqueta`, `ordenar`: etiquetas del diccionario y orden por código para las gráficas.
- `nivel_asimetria`, `comparar_media_mediana`, `rasgos_cluster`: textos de lectura a partir de las cifras calculadas.
- `normalizar_codigo`, `leer_diccionario`, `periodo_de_diccionario`: misma lógica que el notebook de B para leer los diccionarios.

In [ ]:
_CUANTILES = (0.25, 0.5, 0.75, 0.95)


def resumen_numerico(df: DataFrame, cols: list, por: str = None) -> pd.DataFrame:
    # todo en un solo agg; percentile de spark sql es exacto
    qs = ", ".join(str(q) for q in _CUANTILES)
    exprs = []
    for c in cols:
        exprs += [
            F.count(c).alias(f"{c}__n"),
            F.avg(c).alias(f"{c}__media"),
            F.stddev_samp(c).alias(f"{c}__sd"),
            F.min(c).alias(f"{c}__min"),
            F.max(c).alias(f"{c}__max"),
            F.expr(f"percentile(`{c}`, array({qs}))").alias(f"{c}__q"),
        ]
    agregado = (df.groupBy(por) if por else df).agg(*exprs).toPandas()

    filas = []
    for _, r in agregado.iterrows():
        for c in cols:
            q = r[f"{c}__q"]
            p25, p50, p75, p95 = q if isinstance(q, (list, tuple, np.ndarray)) else [np.nan] * 4
            fila = {por: r[por]} if por else {}
            fila.update(variable=c, n=int(r[f"{c}__n"]), media=r[f"{c}__media"], mediana=p50,
                        sd=r[f"{c}__sd"], min=r[f"{c}__min"], max=r[f"{c}__max"],
                        p25=p25, p75=p75, p95=p95)
            filas.append(fila)
    return pd.DataFrame(filas)


def composicion(df: DataFrame, col: str, por: str = None) -> pd.DataFrame:
    # conteo y % por categoría; si hay 'por', el % es dentro de cada grupo
    claves = [por, col] if por else [col]
    t = df.groupBy(*claves).count().toPandas().rename(columns={"count": "n"})
    total = t.groupby(por)["n"].transform("sum") if por else t["n"].sum()
    t["pct"] = 100 * t["n"] / total
    return t.sort_values(claves).reset_index(drop=True)


def histograma(df: DataFrame, col: str, n_bins: int = 60, log10: bool = False) -> pd.DataFrame:
    # los conteos salen de spark con todos los registros; en log10 la columna debe ser > 0
    x = F.log10(F.col(col)) if log10 else F.col(col)
    lo, hi = df.select(F.min(x), F.max(x)).first()
    ancho = (hi - lo) / n_bins if hi > lo else 1.0
    idx = F.least(F.floor((x - F.lit(lo)) / F.lit(ancho)).cast("int"), F.lit(n_bins - 1))
    conteo = (df.groupBy(idx.alias("bin")).count().toPandas()
                .set_index("bin")["count"].reindex(range(n_bins), fill_value=0))
    bordes = lo + ancho * np.arange(n_bins + 1)
    if log10:
        bordes = 10 ** bordes
    return pd.DataFrame({"desde": bordes[:-1], "hasta": bordes[1:], "n": conteo.to_numpy()})


def clave_orden(codigo) -> float:
    # códigos numéricos en orden, DESCONOCIDO u otro texto al final
    s = str(codigo)
    return float(s) if s.lstrip("-").isdigit() else float("inf")


def ordenar(t: pd.DataFrame, col: str) -> pd.DataFrame:
    return t.sort_values(col, key=lambda s: s.map(clave_orden)).reset_index(drop=True)


def etiqueta(codigo, codigos: dict) -> str:
    # el diccionario trae la categoría ocupacional como pregunta ("...gobierno?"), se quita el "?"
    return f"{codigo} - {codigos[codigo].rstrip('?')}" if codigo in codigos else "DESCONOCIDO"


def nivel_asimetria(g: float) -> str:
    # regla práctica usual: |g| < 0.5 casi simétrica, 0.5 a 1 moderada, >= 1 fuerte
    if abs(g) < 0.5:
        return "aproximadamente simétrica"
    lado = "a la derecha" if g > 0 else "a la izquierda"
    return f"{'moderadamente' if abs(g) < 1 else 'fuertemente'} asimétrica {lado}"


def comparar_media_mediana(media: float, mediana: float, tolerancia: float = 0.02) -> str:
    # tolerancia relativa: debajo de 2% se toman como iguales
    if mediana == 0:
        return f"la mediana es 0 y la media {media:,.2f}"
    dif = (media - mediana) / mediana
    if abs(dif) < tolerancia:
        return "la media y la mediana prácticamente coinciden"
    lado, cola = ("por encima", "derecha") if dif > 0 else ("por debajo", "izquierda")
    return f"la media está {abs(dif):.0%} {lado} de la mediana, señal de una cola hacia la {cola}"


UMBRAL_SD = 0.5


def rasgos_cluster(medias: pd.Series, media_global: pd.Series, sd_global: pd.Series,
                   umbral: float = UMBRAL_SD) -> str:
    # distancia de la media del cluster a la global, en desviaciones estándar globales
    partes = []
    for c, m in medias.items():
        z = (m - media_global[c]) / sd_global[c]
        nivel = "alta" if z >= umbral else "baja" if z <= -umbral else "cercana al promedio"
        partes.append(f"{ETIQUETAS_NUM.get(c, c)} {nivel} ({z:+.2f} sd)")
    return "; ".join(partes)


def barras_h(ax, etiquetas, valores, titulo: str, xlabel: str, fmt: str = "{:,.0f}"):
    y = np.arange(len(valores))
    ax.barh(y, valores, color="#2b8cbe")
    ax.set_yticks(y, etiquetas)
    ax.invert_yaxis()
    for i, v in enumerate(valores):
        ax.text(v, i, " " + fmt.format(v), va="center", fontsize=8)
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, max(valores) * 1.2)


# --- diccionarios: misma lógica que el notebook de B
def normalizar_codigo(c) -> str:
    # 1, 1.0, "01", " 1 " -> "1"; texto no numérico queda en mayúsculas
    s = str(c).strip()
    try:
        d = float(s)
        if d == int(d):
            return str(int(d))
    except (ValueError, OverflowError):
        pass
    return s.upper()


def leer_diccionario(ruta: Path, variables: dict) -> dict:
    # {variable original: {código normalizado: etiqueta}} solo para las variables pedidas
    wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    filas = list(wb.worksheets[0].iter_rows(values_only=True))
    wb.close()
    ini = next(i for i, r in enumerate(filas)
               if r[0] and str(r[0]).strip().lower().startswith("valores de variable"))
    valores, actual = {}, None
    for r in filas[ini + 2:]:                       # se salta el encabezado "Valor | Etiqueta"
        if r[0] is not None:
            actual = str(r[0]).strip().upper()
        if actual in variables and r[1] is not None:
            valores.setdefault(actual, {})[normalizar_codigo(r[1])] = str(r[2]).strip()
    return valores


def periodo_de_diccionario(nombre: str):
    m = re.search(r"ENEIC[-_ ]*(IV|III|II|I)[-_ ]*(20\d{2})", nombre, flags=re.I)
    if not m:
        return None
    trimestre = {"I": 1, "II": 2, "III": 3, "IV": 4}[m.group(1).upper()]
    return f"{m.group(2)}T{trimestre}"

### 0.2 Carga de entradas y diccionarios

Se leen `prep_2025` (población analítica de 2025) y `perfiles_2025` (los mismos registros con la columna `cluster`). Se verifica que ambos tengan el mismo número de filas, porque los perfiles se comparan contra las estadísticas globales de `prep_2025`.

Las etiquetas salen de los diccionarios oficiales de 2025, igual que en B. Si los cuatro no coinciden en las tres variables, el notebook se detiene.

In [ ]:
faltan = [str(p) for p in (PREP_2025_DIR, PERFILES_2025_DIR, DICC_DIR) if not p.exists()]
assert not faltan, f"No existen {faltan}. Primero hay que correr los notebooks de A y B."

df_2025 = spark.read.parquet(str(PREP_2025_DIR)).persist()
perfiles = spark.read.parquet(str(PERFILES_2025_DIR)).persist()

faltan_cols = (set(VARS_NUM + VARS_CAT + ["periodo_archivo"]) - set(df_2025.columns)) \
            | (set(VARS_NUM + VARS_CAT + ["cluster"]) - set(perfiles.columns))
assert not faltan_cols, f"Faltan columnas: {sorted(faltan_cols)}"

n_2025 = df_2025.count()
assert perfiles.count() == n_2025, "perfiles_2025 y prep_2025 no tienen el mismo número de filas"
print(f"Registros de la población analítica 2025: {n_2025:,}")

In [ ]:
dicc = {}
for ruta in sorted(DICC_DIR.glob("*.xlsx")):
    p = periodo_de_diccionario(ruta.name)
    if p in PERIODOS_2025:
        dicc[p] = leer_diccionario(ruta, VARIABLES_CAT)

faltan = [p for p in PERIODOS_2025 if p not in dicc]
assert not faltan, f"No se encontró diccionario para {faltan} en {DICC_DIR}"
ref = dicc[PERIODOS_2025[0]]
assert all(d == ref for d in dicc.values()), "Los diccionarios de 2025 difieren en las variables categóricas"

CODIGOS = {VARIABLES_CAT[v]: ref[v] for v in VARIABLES_CAT}
print("Códigos por variable:", {c: len(d) for c, d in CODIGOS.items()})

---
## 2. Estadística descriptiva y preguntas de exploración

### 2.1 Estadísticas de las variables numéricas (2025, todos los registros)

In [ ]:
resumen_2025 = resumen_numerico(df_2025, VARS_NUM).set_index("variable")
display(resumen_2025.rename(index=ETIQUETAS_NUM).round(2))

sal, ed, an, hr = (resumen_2025.loc[c] for c in VARS_NUM)
display(Markdown(
    f"**Lectura.** Se analizan {int(sal['n']):,} registros de 2025. "
    f"El salario mensual tiene mediana de Q{sal['mediana']:,.0f} y media de Q{sal['media']:,.0f} "
    f"({comparar_media_mediana(sal['media'], sal['mediana'])}). La mitad central está entre "
    f"Q{sal['p25']:,.0f} (P25) y Q{sal['p75']:,.0f} (P75), el P95 es Q{sal['p95']:,.0f} y el máximo "
    f"Q{sal['max']:,.0f}, {sal['max'] / sal['mediana']:,.1f} veces la mediana; la desviación estándar es "
    f"Q{sal['sd']:,.0f}. "
    f"La edad mediana es {ed['mediana']:.0f} años (P25–P75: {ed['p25']:.0f}–{ed['p75']:.0f}; "
    f"rango {ed['min']:.0f}–{ed['max']:.0f}). "
    f"La antigüedad mediana es {an['mediana']:.1f} años y la media {an['media']:.1f} "
    f"({comparar_media_mediana(an['media'], an['mediana'])}); el P95 es {an['p95']:.1f} años. "
    f"Las horas habituales tienen mediana {hr['mediana']:.0f} por semana (P25–P75: "
    f"{hr['p25']:.0f}–{hr['p75']:.0f}; rango {hr['min']:.0f}–{hr['max']:.0f})."
))

### 2.2 ¿Cómo se distribuyen los registros entre categorías ocupacionales, niveles educativos y dominios?

In [ ]:
dist_cat = {c: ordenar(composicion(df_2025, c), c) for c in VARS_CAT}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, c in zip(axes, VARS_CAT):
    t = dist_cat[c]
    barras_h(ax, [etiqueta(v, CODIGOS[c]) for v in t[c]], t["pct"],
             ETIQUETAS_CAT[c], "% de registros", "{:.1f}%")
fig.suptitle(f"Distribución de registros - población analítica 2025 (n = {n_2025:,})", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_distribucion_categorias.png", dpi=120, bbox_inches="tight")
plt.show()

lineas = []
for c in VARS_CAT:
    t = dist_cat[c].sort_values("n", ascending=False)
    mayor, menor = t.iloc[0], t.iloc[-1]
    desc = t.loc[t[c] == "DESCONOCIDO", "pct"].sum()
    lineas.append(
        f"- **{ETIQUETAS_CAT[c]}** ({len(t)} categorías): la más frecuente es {etiqueta(mayor[c], CODIGOS[c])} "
        f"con {mayor['pct']:.1f}% ({int(mayor['n']):,} registros) y la menos frecuente "
        f"{etiqueta(menor[c], CODIGOS[c])} con {menor['pct']:.1f}% ({int(menor['n']):,}). "
        f"`DESCONOCIDO`: {desc:.1f}%."
    )
display(Markdown("**Lectura.**\n\n" + "\n".join(lineas) +
                 "\n\nSon conteos de registros sin ponderar: describen la muestra analizada, no la composición "
                 "del mercado laboral del país (para eso habría que ponderar con `FACTOR`)."))

### 2.3 ¿El salario presenta una distribución simétrica o asimétrica?

Los histogramas se calculan en Spark con todos los registros (a pandas solo pasa la tabla de intervalos). El de la derecha usa **escala logarítmica base 10 en el eje X**, solo para visualizar; el objetivo del modelado sigue siendo el salario en quetzales.

In [ ]:
h_lin = histograma(df_2025, "salario_mensual", n_bins=60)
h_log = histograma(df_2025, "salario_mensual", n_bins=60, log10=True)
asimetria = df_2025.select(F.skewness("salario_mensual")).first()[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, h in zip(axes, (h_lin, h_log)):
    ax.bar(h["desde"], h["n"], width=h["hasta"] - h["desde"], align="edge",
           color="#2b8cbe", edgecolor="white", linewidth=0.3)
    ax.axvline(sal["media"], color="#e6550d", ls="--", lw=1.3, label=f"media Q{sal['media']:,.0f}")
    ax.axvline(sal["mediana"], color="#31a354", lw=1.3, label=f"mediana Q{sal['mediana']:,.0f}")
    ax.set_ylabel("Registros")
    ax.legend()
axes[0].set_xlabel("Salario mensual (Q) - escala lineal")
axes[0].set_title("Salario mensual, escala lineal")
axes[1].set_xscale("log")
axes[1].set_xlabel("Salario mensual (Q) - ESCALA LOGARÍTMICA (base 10)")
axes[1].set_title("Salario mensual, eje X logarítmico")
fig.suptitle(f"Distribución del salario mensual - 2025 (n = {n_2025:,}, todos los registros)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_histograma_salario.png", dpi=120, bbox_inches="tight")
plt.show()

pico = h_lin.loc[h_lin["n"].idxmax()]
display(Markdown(
    f"**Lectura.** El coeficiente de asimetría (`skewness` de Spark) es {asimetria:.2f}: la distribución es "
    f"**{nivel_asimetria(asimetria)}** (regla práctica: |g| < 0.5 casi simétrica, 0.5–1 moderada, ≥ 1 fuerte). "
    f"En escala lineal el intervalo con más registros va de Q{pico['desde']:,.0f} a Q{pico['hasta']:,.0f}, "
    f"mientras el eje se extiende hasta Q{sal['max']:,.0f}. La escala logarítmica comprime esa cola y deja ver "
    f"la forma de la parte central de la distribución."
))

### 2.4 ¿Qué diferencia existe entre la media y la mediana del salario?

In [ ]:
dif_abs = sal["media"] - sal["mediana"]
display(pd.DataFrame({"media": [sal["media"]], "mediana": [sal["mediana"]], "diferencia (Q)": [dif_abs],
                      "diferencia (% de la mediana)": [100 * dif_abs / sal["mediana"]]}).round(2))

display(Markdown(
    f"**Lectura.** La media es Q{sal['media']:,.0f} y la mediana Q{sal['mediana']:,.0f}: "
    f"{comparar_media_mediana(sal['media'], sal['mediana'])} (diferencia de Q{dif_abs:,.0f}). "
    "La mediana es el valor que deja la mitad de los registros a cada lado y no se mueve con unos pocos salarios "
    "muy altos; la media sí, porque suma todos los montos. Cuando la media supera a la mediana, un registro "
    "\"típico\" gana menos que el promedio, y la mediana describe mejor el salario central. Para el modelado "
    "esto importa: un modelo que minimiza errores cuadráticos (RMSE) se deja influir por esa cola, y un baseline "
    "que predice la media no es lo mismo que uno que predice la mediana."
))

### 2.5 ¿Cómo varía el salario mediano entre niveles educativos y categorías ocupacionales?

In [ ]:
sal_por = {c: ordenar(resumen_numerico(df_2025, ["salario_mensual"], por=c), c)
           for c in ["nivel_educativo", "categoria_ocupacional"]}

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
for ax, (c, t) in zip(axes, sal_por.items()):
    etiquetas = [f"{etiqueta(v, CODIGOS[c])} (n={n:,})" for v, n in zip(t[c], t["n"])]
    barras_h(ax, etiquetas, t["mediana"], f"Salario mediano por {ETIQUETAS_CAT[c].lower()}",
             "Salario mensual mediano (Q)", "Q{:,.0f}")
    ax.axvline(sal["mediana"], color="gray", ls="--", lw=1, label=f"mediana global Q{sal['mediana']:,.0f}")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_salario_mediano_grupos.png", dpi=120, bbox_inches="tight")
plt.show()

lineas = []
for c, t in sal_por.items():
    display(t[[c, "n", "mediana", "media", "p25", "p75"]]
            .assign(etiqueta=[etiqueta(v, CODIGOS[c]) for v in t[c]]).round(0))
    alto, bajo = t.loc[t["mediana"].idxmax()], t.loc[t["mediana"].idxmin()]
    lineas.append(
        f"- **{ETIQUETAS_CAT[c]}:** la mediana más alta es la de {etiqueta(alto[c], CODIGOS[c])} "
        f"(Q{alto['mediana']:,.0f}, n = {int(alto['n']):,}) y la más baja la de "
        f"{etiqueta(bajo[c], CODIGOS[c])} (Q{bajo['mediana']:,.0f}, n = {int(bajo['n']):,}); "
        f"la primera es {alto['mediana'] / bajo['mediana']:.1f} veces la segunda."
    )
display(Markdown("**Lectura.**\n\n" + "\n".join(lineas) +
                 "\n\nLos grupos con pocos registros (ver n) tienen medianas menos estables. Son diferencias "
                 "descriptivas entre registros: no aíslan el efecto de la educación o la categoría, porque "
                 "los grupos también difieren en edad, antigüedad, jornada y dominio."))

### 2.6 ¿Cómo cambian el tamaño de la muestra analítica y el salario mediano entre trimestres?

El trimestre se toma de `periodo_archivo` (asignado desde el archivo de procedencia), no de `TRIMESTRE`.

In [ ]:
por_trim = resumen_numerico(df_2025, ["salario_mensual"], por="periodo_archivo").sort_values("periodo_archivo")
display(por_trim[["periodo_archivo", "n", "mediana", "media", "p25", "p75"]].round(0))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
a1.bar(por_trim["periodo_archivo"], por_trim["n"], color="#2b8cbe")
for x, v in zip(por_trim["periodo_archivo"], por_trim["n"]):
    a1.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
a1.set_title("Registros analíticos por trimestre")
a1.set_ylabel("Registros")
a2.plot(por_trim["periodo_archivo"], por_trim["mediana"], "o-", color="#e6550d")
for x, v in zip(por_trim["periodo_archivo"], por_trim["mediana"]):
    a2.annotate(f"Q{v:,.0f}", (x, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
a2.set_title("Salario mensual mediano por trimestre")
a2.set_ylabel("Q")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_trimestres.png", dpi=120, bbox_inches="tight")
plt.show()

n_min, n_max = por_trim.loc[por_trim["n"].idxmin()], por_trim.loc[por_trim["n"].idxmax()]
primero, ultimo = por_trim.iloc[0], por_trim.iloc[-1]
display(Markdown(
    f"**Lectura.** El tamaño de la muestra analítica va de {int(n_min['n']):,} registros "
    f"({n_min['periodo_archivo']}) a {int(n_max['n']):,} ({n_max['periodo_archivo']}), una diferencia de "
    f"{100 * (n_max['n'] - n_min['n']) / n_min['n']:.1f}%. El salario mediano pasa de Q{primero['mediana']:,.0f} "
    f"en {primero['periodo_archivo']} a Q{ultimo['mediana']:,.0f} en {ultimo['periodo_archivo']} "
    f"({100 * (ultimo['mediana'] - primero['mediana']) / primero['mediana']:+.1f}%), con mínimo "
    f"Q{por_trim['mediana'].min():,.0f} y máximo Q{por_trim['mediana'].max():,.0f} en el año. "
    "Como la muestra rota y no está ponderada, estas variaciones describen los registros de cada archivo y no "
    "deben leerse como la evolución oficial de los salarios."
))

### 2.7 Primera discusión sobre salarios extremos

Los salarios extremos **se mantienen** (el enunciado prohíbe recortarlos). Aquí solo se cuantifica cuántos hay y cuánto pesan en la media. La media "hasta P95" es un cálculo descriptivo, no un cambio en los datos.

In [ ]:
cerca_sup = sal["p75"] + 1.5 * (sal["p75"] - sal["p25"])
s = F.col("salario_mensual")
ext = df_2025.agg(
    F.sum((s > sal["p95"]).cast("int")).alias("n_sobre_p95"),
    F.sum((s > cerca_sup).cast("int")).alias("n_sobre_cerca"),
    F.sum(F.when(s > sal["p95"], s)).alias("masa_sobre_p95"),
    F.sum(s).alias("masa_total"),
    F.avg(F.when(s <= sal["p95"], s)).alias("media_hasta_p95"),
).first()

display(pd.DataFrame([{
    "P95 (Q)": sal["p95"], "registros > P95": ext["n_sobre_p95"],
    "límite P75 + 1.5·IQR (Q)": cerca_sup, "registros > límite IQR": ext["n_sobre_cerca"],
    "% del total de salarios que suman los > P95": 100 * ext["masa_sobre_p95"] / ext["masa_total"],
    "media con todos (Q)": sal["media"], "media de los ≤ P95 (Q)": ext["media_hasta_p95"],
}]).round(2).T.rename(columns={0: "valor"}))

display(Markdown(
    f"**Lectura.** {ext['n_sobre_cerca']:,} registros ({100 * ext['n_sobre_cerca'] / n_2025:.1f}%) superan el "
    f"límite P75 + 1.5·IQR (Q{cerca_sup:,.0f}), y los {ext['n_sobre_p95']:,} que están sobre el P95 concentran "
    f"el {100 * ext['masa_sobre_p95'] / ext['masa_total']:.1f}% de la suma de salarios. Con todos los registros la "
    f"media es Q{sal['media']:,.0f}; sin contar lo que está sobre el P95 sería Q{ext['media_hasta_p95']:,.0f}. "
    "Se mantienen porque pueden ser salarios reales (puestos directivos o especializados) y el enunciado pide "
    "evaluarlos. Su influencia esperada: inflan la media y la desviación estándar, pueden pesar en las "
    "correlaciones de Pearson y, en los modelos, dominan el RMSE más que el MAE, así que conviene reportar ambos "
    "y revisar los errores por tramo de salario en el Ejercicio 8."
))

---
## 4. Segmentación KMeans - perfiles de los clusters

Se usan los clusters que asignó Persona B (`perfiles_2025`, variante y K elegidos en su sección 4.1). Aquí se describen **en la escala original** de cada variable, no en la estandarizada. Según el notebook de B la variante elegida es `sin_salario`, así que el salario no se usó para agrupar y compararlo entre clusters sirve de validación externa; si B cambia de variante, hay que revisar este párrafo.

### 4.4 Tamaño de cada cluster

In [ ]:
tam = composicion(perfiles, "cluster").set_index("cluster")
CLUSTERS = tam.index.tolist()
K = len(CLUSTERS)
print(f"K = {K}; clusters: {CLUSTERS}")
display(tam.round(1))

grande, chico = tam["n"].idxmax(), tam["n"].idxmin()
display(Markdown(
    f"**Lectura.** Los {K} clusters suman {int(tam['n'].sum()):,} registros. El más grande es el {grande} "
    f"({tam.loc[grande, 'pct']:.1f}%) y el más pequeño el {chico} ({tam.loc[chico, 'pct']:.1f}%)."
))

### 4.5 Perfil numérico por cluster (escala original)

Para cada variable se compara la media del cluster con la media global en **desviaciones estándar globales**, la misma escala que usó KMeans al estandarizar. Se marca como "alta" o "baja" cuando la diferencia es de al menos ±0.5 sd; por debajo de eso, "cercana al promedio". Es un umbral de apoyo para leer los perfiles, no una prueba estadística.

In [ ]:
perfil = resumen_numerico(perfiles, VARS_NUM, por="cluster")
medias = perfil.pivot(index="cluster", columns="variable", values="media")[VARS_NUM]
medianas = perfil.pivot(index="cluster", columns="variable", values="mediana")[VARS_NUM]

tabla_perfil = (tam[["n", "pct"]]
                .join(medianas.rename(columns=lambda c: f"mediana {c}"))
                .join(medias.rename(columns=lambda c: f"media {c}")))
display(tabla_perfil.round(1))

VARS_KM = ["edad", "antiguedad", "horas_semanales"]
rasgos = {k: rasgos_cluster(medias.loc[k, VARS_KM], resumen_2025["media"], resumen_2025["sd"]) for k in CLUSTERS}
sal_alto, sal_bajo = medianas["salario_mensual"].idxmax(), medianas["salario_mensual"].idxmin()

display(Markdown(
    "**Lectura.**\n\n" +
    "\n".join(f"- **Cluster {k}** ({tam.loc[k, 'pct']:.1f}%): {rasgos[k]}. Salario mediano "
              f"Q{medianas.loc[k, 'salario_mensual']:,.0f}." for k in CLUSTERS) +
    f"\n\nEl salario mediano más alto es el del cluster {sal_alto} (Q{medianas.loc[sal_alto, 'salario_mensual']:,.0f}) "
    f"y el más bajo el del cluster {sal_bajo} (Q{medianas.loc[sal_bajo, 'salario_mensual']:,.0f}), frente a una "
    f"mediana global de Q{sal['mediana']:,.0f}. Son asociaciones entre perfiles y salario, no efectos causales."
))

In [ ]:
fig, axes = plt.subplots(1, len(VARS_NUM), figsize=(18, 4.2))
paleta = sns.color_palette("tab10", K)
for ax, c in zip(axes, VARS_NUM):
    ax.bar([f"C{k}" for k in CLUSTERS], medias[c], color=paleta)
    ax.axhline(resumen_2025.loc[c, "media"], color="gray", ls="--", lw=1, label="media global")
    for i, v in enumerate(medias[c]):
        ax.text(i, v, f"{v:,.1f}" if c != "salario_mensual" else f"Q{v:,.0f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(f"Media de {ETIQUETAS_NUM[c].lower()}")
axes[0].legend()
fig.suptitle(f"Perfil de los clusters (medias en escala original, todos los registros de 2025, K = {K})", y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_perfil_medias.png", dpi=120, bbox_inches="tight")
plt.show()

### 4.6 Dispersión por cluster (muestra de hasta 5,000 registros)

Solo para visualizar la separación; todas las cifras de esta sección se calcularon con todos los registros.

In [ ]:
muestra = (perfiles.select(*VARS_KM, "cluster")
           .sample(fraction=min(1.0, 1.3 * MAX_FILAS_GRAFICO / n_2025), seed=SEED)
           .limit(MAX_FILAS_GRAFICO)
           .toPandas())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (x, y) in zip(axes, [("edad", "antiguedad"), ("edad", "horas_semanales")]):
    sns.scatterplot(data=muestra, x=x, y=y, hue="cluster", palette=paleta, s=10, alpha=0.5,
                    linewidth=0, ax=ax)
    ax.set_xlabel(ETIQUETAS_NUM[x])
    ax.set_ylabel(ETIQUETAS_NUM[y])
fig.suptitle(f"Muestra aleatoria de {len(muestra):,} registros coloreados por cluster (semilla {SEED})", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_dispersion_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

### 4.7 Composición de cada cluster por categoría ocupacional, nivel educativo y dominio

Porcentaje de los registros de cada cluster en cada categoría. Para cada cluster se señala la categoría más **sobrerrepresentada** respecto de su peso en toda la población analítica (tabla 2.2), en puntos porcentuales.

In [ ]:
comp = {c: composicion(perfiles, c, por="cluster") for c in VARS_CAT}

fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
for ax, c in zip(axes, VARS_CAT):
    tabla = comp[c].pivot(index="cluster", columns=c, values="pct").fillna(0)
    tabla = tabla[sorted(tabla.columns, key=clave_orden)]
    tabla.columns = [etiqueta(v, CODIGOS[c]) for v in tabla.columns]
    display(Markdown(f"**{ETIQUETAS_CAT[c]}** (% dentro de cada cluster)"))
    display(tabla.round(1))
    tabla.plot(kind="barh", stacked=True, ax=ax, colormap="tab20", width=0.8, legend=False)
    ax.set_title(ETIQUETAS_CAT[c])
    ax.set_xlabel("% de registros del cluster")
    ax.invert_yaxis()
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_composicion_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

sobre = {}
for c in VARS_CAT:
    glob = dist_cat[c].set_index(c)["pct"]
    t = comp[c].assign(dif=lambda d: d["pct"] - d[c].map(glob))
    for k, g in t.groupby("cluster"):
        r = g.loc[g["dif"].idxmax()]
        sobre.setdefault(k, []).append(f"{etiqueta(r[c], CODIGOS[c])} ({r['pct']:.1f}%, {r['dif']:+.1f} pp)")

display(Markdown("**Lectura: categoría más sobrerrepresentada en cada cluster** "
                 "(categoría ocupacional · nivel educativo · dominio).\n\n" +
                 "\n".join(f"- **Cluster {k}:** " + " · ".join(sobre[k]) for k in CLUSTERS)))

### 4.8 Nombre y descripción de cada cluster

Tabla de apoyo con los rasgos calculados arriba. Con ella el equipo asigna un nombre y una descripción a cada cluster en `NOMBRES_CLUSTER` (por ejemplo, "jóvenes de jornada completa con poca antigüedad"). La celda se detiene si algún cluster queda sin nombre, para que no se entregue incompleta.

In [ ]:
apoyo = pd.DataFrame({
    "% registros": tam["pct"].round(1),
    "rasgos (edad, antigüedad, horas)": pd.Series(rasgos),
    "salario mediano (Q)": medianas["salario_mensual"].round(0),
    "más sobrerrepresentado": pd.Series({k: " · ".join(v) for k, v in sobre.items()}),
})
with pd.option_context("display.max_colwidth", None):
    display(apoyo)

In [ ]:
# nombres puestos viendo la tabla de apoyo (corrida con K = 2 de B); si B cambia K o variante, revisar
NOMBRES_CLUSTER = {
    0: ("Jóvenes con poca antigüedad en su empleo",
        "72.5% de los registros. Edad mediana 28 años, antigüedad mediana 1.3 años y 46 horas semanales "
        "(jornada cercana al promedio). Más peso de empleados de empresa privada y de nivel diversificado. "
        "Salario mediano Q3,000, igual al global."),
    1: ("Adultos con trayectoria larga en su empleo",
        "27.5% de los registros. Edad mediana 51 años y antigüedad mediana 13 años, con 40 horas "
        "semanales (jornada cercana al promedio, aunque un poco menor). Sobrerrepresenta empleados de gobierno (+11 pp) y personas sin "
        "escolaridad (+7 pp): mezcla trabajadores públicos estables con trabajadores mayores de baja escolaridad. "
        "Salario mediano Q3,466, el más alto de los dos."),
}

sin_nombre = [k for k in CLUSTERS if k not in NOMBRES_CLUSTER]
assert not sin_nombre, f"Faltan nombres para los clusters {sin_nombre}: llenar NOMBRES_CLUSTER con la tabla de apoyo"

tabla_nombres = pd.DataFrame(
    [(k, *NOMBRES_CLUSTER[k], tam.loc[k, "pct"], medianas.loc[k, "salario_mensual"]) for k in CLUSTERS],
    columns=["cluster", "nombre", "descripción", "% registros", "salario mediano (Q)"],
).round(1)
with pd.option_context("display.max_colwidth", None):
    display(tabla_nombres)

---
## Anexo. Prueba de las funciones de apoyo con datos sintéticos

Datos artificiales con resultados conocidos, comprobados con `assert`: percentiles exactos frente a `numpy`, porcentajes que suman 100, histogramas que conservan todos los registros, etiquetas (el código educativo `0` se etiqueta como categoría válida y un código fuera del diccionario como `DESCONOCIDO`), textos de lectura y lectura de un diccionario con la misma estructura que los oficiales. No depende de los datos reales.

In [ ]:
import tempfile

# resumen_numerico: global y por grupo contra numpy (interpolación lineal, sd muestral)
vals_a, vals_b = [float(v) for v in range(1, 11)], [5.0, 7.0, 100.0]
sint = spark.createDataFrame([("a", v) for v in vals_a] + [("b", v) for v in vals_b], ["g", "x"])

def _chequear(fila, vals):
    esperado = dict(zip(["p25", "mediana", "p75", "p95"], np.percentile(vals, [25, 50, 75, 95])))
    esperado.update(n=len(vals), media=np.mean(vals), sd=np.std(vals, ddof=1), min=min(vals), max=max(vals))
    for k, v in esperado.items():
        assert np.isclose(fila[k], v), f"{k}: {fila[k]} != {v}"

_chequear(resumen_numerico(sint, ["x"]).iloc[0], vals_a + vals_b)
por_g = resumen_numerico(sint, ["x"], por="g").set_index("g")
_chequear(por_g.loc["a"], vals_a)
_chequear(por_g.loc["b"], vals_b)

# composicion: % suma 100 global y dentro de cada grupo
assert np.isclose(composicion(sint, "g")["pct"].sum(), 100)
cats = spark.createDataFrame([(1, "x"), (1, "y"), (1, "y"), (2, "x")], ["k", "c"])
assert np.allclose(composicion(cats, "c", por="k").groupby("k")["pct"].sum(), 100)

# histograma: conserva todos los registros, bordes crecientes; también con columna constante
for log in (False, True):
    h = histograma(sint, "x", n_bins=7, log10=log)
    assert h["n"].sum() == sint.count() and (np.diff(h["desde"]) > 0).all()
assert histograma(spark.createDataFrame([(3.0,), (3.0,)], ["x"]), "x", n_bins=4)["n"].sum() == 2

# etiquetas y orden
codigos = {"0": "NINGUNO", "1": "PRIMARIA", "2": "Empleado de gobierno?"}
assert etiqueta("0", codigos) == "0 - NINGUNO" and etiqueta("2", codigos) == "2 - Empleado de gobierno"
assert etiqueta("99", codigos) == "DESCONOCIDO" and etiqueta("DESCONOCIDO", codigos) == "DESCONOCIDO"
orden = ordenar(pd.DataFrame({"c": ["10", "DESCONOCIDO", "2", "0"]}), "c")["c"].tolist()
assert orden == ["0", "2", "10", "DESCONOCIDO"], orden

# textos de lectura
assert nivel_asimetria(0.2) == "aproximadamente simétrica"
assert nivel_asimetria(0.7) == "moderadamente asimétrica a la derecha"
assert nivel_asimetria(-3) == "fuertemente asimétrica a la izquierda"
assert "coinciden" in comparar_media_mediana(100, 101)
assert "por encima" in comparar_media_mediana(150, 100) and "mediana es 0" in comparar_media_mediana(1, 0)
r = rasgos_cluster(pd.Series({"edad": 50.0, "horas_semanales": 40.0}),
                   pd.Series({"edad": 40.0, "horas_semanales": 42.0}), pd.Series({"edad": 10.0, "horas_semanales": 10.0}))
assert "Edad (años) alta (+1.00 sd)" in r and "Horas semanales cercana al promedio" in r, r

# diccionario: misma estructura que los oficiales (bloque tras "Valores de variable", códigos como número o texto)
assert normalizar_codigo("01") == "1" and normalizar_codigo(1.0) == "1" and normalizar_codigo(" a ") == "A"
assert periodo_de_diccionario("Diccionario ENEIC-IV-2025.xlsx") == "2025T4"
with tempfile.TemporaryDirectory() as tmp:
    ruta = Path(tmp) / "dicc.xlsx"
    wb = openpyxl.Workbook()
    for fila in [("Información de variable",), ("Valores de variable",), ("Variable", "Valor", "Etiqueta"),
                 ("P03A03A", 0, "NINGUNO"), (None, "01", "PRIMARIA"), ("OTRA", 1, "X"), ("DOMINIO", 1.0, "URBANO")]:
        wb.active.append(fila)
    wb.save(ruta)
    leido = leer_diccionario(ruta, VARIABLES_CAT)
assert leido == {"P03A03A": {"0": "NINGUNO", "1": "PRIMARIA"}, "DOMINIO": {"1": "URBANO"}}, leido

print("OK: todas las funciones de apoyo se comportan como se esperaba.")

In [ ]:
_ = df_2025.unpersist()
_ = perfiles.unpersist()